# ZWO Camera GUI — External Client Example

The GUI embeds an optional JSON-over-WebSocket server. Launch it with a port:

```bash
python -m zwo_camera_gui --ws-port 8765
```

Then drive it from anywhere with `ASIClient`. The client holds one persistent
connection, handles JSON framing, and blocks `record()` until the FITS save
actually finishes.

In [12]:
from zwo_camera_gui.client import ASIClient

cam = ASIClient("ws://localhost:8765")
cam.connect()

## Attach to a camera

The WebSocket connection and the *camera* connection are separate: opening
the client just talks to the GUI, it doesn't automatically open a camera.
Enumerate what's on the bus and attach to one before doing anything else.
This is equivalent to clicking **Refresh** then **Connect** in the GUI
sidebar — the GUI's combo box will update to match.

In [13]:
print(cam.list_cameras())

cam.connect_camera(index=0)

[{'index': 0, 'name': 'ZWO ASI662MM'}]


{'cmd': 'connect_camera', 'ok': True, 'camera': 'ZWO ASI662MM'}

## Status

Dumps the control snapshot — useful for seeing exactly what the attached
camera exposes (the ASI662MM has no real Offset register, the ASI990MM Pro
has independent `FrameRateLimit`, etc.).

In [14]:
import json
print(json.dumps(cam.status(), indent=2))

{
  "cmd": "status",
  "connected": true,
  "streaming": false,
  "camera": "ZWO ASI662MM",
  "controls": {
    "Gain": 200,
    "Exposure": 10000,
    "Offset": 15,
    "BandWidth": 100,
    "HighSpeedMode": 0,
    "Flip": 0,
    "HardwareBin": 0,
    "AutoExpMaxExpMS": 100
  }
}


## Configure the camera

Keys match the control names from `status()`. **Exposure is in microseconds**
(`50_000` µs = 50 ms).

In [15]:
cam.set(
    Exposure=125_000,   # 125 ms
    Gain=200,
    Offset=15,
    Flip=0,             # 0=None, 1=H, 2=V, 3=Both
    HighSpeedMode=0,
    img_type="RAW16",
)

{'cmd': 'set', 'ok': True}

## Capture some frames

`capture_frames(n)` is the quick path: it starts the stream if it isn't already running, captures `n` frames into a FITS file, and blocks until the save completes. It leaves the stream running so the next call is instant and a GUI operator keeps their live preview.

- `stack=True` (default) → one cube FITS, `basename.fits`
- `stack=False` → one file per frame, `basename_0000.fits`, `basename_0001.fits`, ...

All current camera controls land in each FITS header (INSTRUME, NFRAMES, STRMFPS, ELAPSED, DEPTH, ROI, cooler temp, plus every writable control). `capture_frames()` raises `ASIClientError` if the save fails.

In [16]:
done = cam.capture_frames(20, directory="./captures", basename="demo_stack")
print(done["message"])

#done = cam.capture_frames(5, directory="./captures", basename="demo_indiv", stack=False)
#print(done["message"])

Saved 20 frames -> ./captures\demo_stack.fits  (82.9 MB, 7.8 fps)


## OBSTYPE and custom header entries

`capture_frames()` accepts an `obstype` string (written to `OBSTYPE`) and an
`extra_headers` argument for arbitrary FITS keywords. `extra_headers` accepts
three shapes — use whichever is convenient:

- **dict** — value alone, or `(value, comment)` tuple as the value
- **iterable of tuples** — `(key, value)` or `(key, value, comment)`
- **iterable of `astropy.io.fits.Card`** objects

Extras are applied after the auto-filled metadata, so they can override
anything the GUI set (including `EXPTIME`).

In [ ]:
from astropy.io.fits import Card

# dict form — values alone, or (value, comment) tuples
cam.capture_frames(
    5,
    directory="./captures",
    basename="dark",
    obstype="DARK",
    extra_headers={
        "OBJECT": "bias frame set",
        "FILTER": ("None", "shutter closed"),
    },
)

# iterable form — mix tuples and astropy Cards
cam.capture_frames(
    5,
    directory="./captures",
    basename="light",
    obstype="LIGHT",
    extra_headers=[
        ("OBJECT", "M42"),
        Card("FILTER", "Halpha", "narrowband filter"),
    ],
)

## Batch capture — exposure sweep

`capture_frames()` auto-starts the stream if it isn't already running and leaves it running afterward, so the sweep loop is a one-liner per point. Pass `stack=False` to write one file per frame instead of a cube.

In [ ]:
for exp_ms in [1, 10, 100, 500]:
    cam.set(Exposure=int(exp_ms * 1000))
    done = cam.capture_frames(
        10,
        directory="./captures/sweep",
        basename=f"exp_{exp_ms:g}ms",
    )
    print(f"{exp_ms:>6g} ms -> {done['message']}")

## Manual stream control (optional)

`capture_frames()` is enough for most workflows. When you need finer control — e.g. warm up the stream for several seconds before the first capture, or stop it between captures to save USB bandwidth — drive it yourself with `start_stream()` / `record()` / `stop_stream()`. `record()` does **not** auto-start the stream and will fail if nothing is streaming.

In [ ]:
cam.stop_stream()

cam.start_stream()
# ...do something that benefits from a warm stream...
done = cam.record(10, directory="./captures", basename="manual")
print(done["message"])
cam.stop_stream()

## Cooler (cooled cameras only)

In [ ]:
cam.cooler(on=True, target=-10)

## Shut down

The client is also a context manager, so for one-shot scripts you can skip
the explicit `connect()` / `close()`:

```python
with ASIClient() as cam:
    cam.connect_camera(0)
    cam.set(Exposure=10_000)
    cam.capture_frames(50, basename="shot")
```

In [ ]:
cam.stop_stream()
cam.close()